# Feature engineering for development permit approval prediction

This notebook prepares the feature set for predicting whether a Calgary
development permit will be approved. We apply TF-IDF vectorisation on
permit descriptions, encode categorical columns, extract temporal
features, and combine all components into a single feature matrix.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, '..')
from src.data_loader import load_and_preprocess, clean_text, APPROVED_STATUSES
from src.model import FeatureBuilder, CATEGORICAL_COLS, NUMERICAL_COLS

pd.set_option('display.max_columns', 50)

## 1. Load and preprocess permits

In [ ]:
df = load_and_preprocess(use_cache=True)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Target distribution
print("Target column: approved")
print(df['approved'].value_counts())
print(f"\nApproval rate: {df['approved'].mean():.2%}")

## 2. Class balance check

An imbalanced target can bias models toward the majority class. We
check the ratio before deciding whether to apply sampling strategies.

In [ ]:
class_counts = df['approved'].value_counts()
ratio = class_counts.min() / class_counts.max()

fig = px.bar(
    x=['Denied (0)', 'Approved (1)'],
    y=[class_counts.get(0, 0), class_counts.get(1, 0)],
    title='Target class distribution',
    labels={'x': 'Class', 'y': 'Count'}
)
fig.update_layout(height=350)
fig.show()

print(f"Minority/majority ratio: {ratio:.3f}")
if ratio < 0.3:
    print("Significant class imbalance detected. Consider stratified splits and class weighting.")
else:
    print("Class balance is acceptable for standard training.")

## 3. TF-IDF vectorisation on descriptions

The `description_clean` column was created during preprocessing by
lowering case, removing HTML tags, and stripping special characters.
We now convert it to TF-IDF features using bigrams.

In [ ]:
# Sample cleaned descriptions
print("Sample cleaned descriptions:")
for i, text in enumerate(df['description_clean'].dropna().head(5)):
    print(f"  [{i}] {text[:120]}...")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=500, stop_words='english',
    ngram_range=(1, 2), min_df=5, max_df=0.95
)
corpus = df['description_clean'].fillna('').astype(str)
tfidf_matrix = tfidf.fit_transform(corpus)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"\nTop 20 terms by document frequency:")
terms = tfidf.get_feature_names_out()
doc_freq = (tfidf_matrix > 0).sum(axis=0).A1
top_idx = doc_freq.argsort()[-20:][::-1]
for idx in top_idx:
    print(f"  {terms[idx]}: {doc_freq[idx]} docs")

## 4. Encode categorical features

The categorical columns -- category, landusedistrict, communityname,
quadrant, and permitteddiscretionary -- are label-encoded.

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in CATEGORICAL_COLS:
    if col in df.columns:
        n_unique = df[col].nunique()
        print(f"{col}: {n_unique} unique values")
        print(f"  Top 5: {df[col].value_counts().head(5).to_dict()}")

In [ ]:
# Encode using the FeatureBuilder from src/model.py
fb = FeatureBuilder(tfidf_max_features=500)
fb.fit(df)

print(f"Label encoders fitted for: {list(fb.label_encoders.keys())}")
for col, le in fb.label_encoders.items():
    print(f"  {col}: {len(le.classes_)} classes")

## 5. Temporal features

The preprocessing step extracted `applied_year`, `applied_month`, and
`applied_day_of_week` from the application date.

In [ ]:
print(f"Temporal features: {NUMERICAL_COLS}")
print(df[NUMERICAL_COLS].describe())

# Approval rate by year
yearly = df.groupby('applied_year')['approved'].mean().reset_index()
fig = px.line(
    yearly, x='applied_year', y='approved',
    title='Approval rate by year',
    labels={'applied_year': 'Year', 'approved': 'Approval rate'},
    markers=True
)
fig.update_yaxes(tickformat='.0%')
fig.update_layout(height=350)
fig.show()

## 6. Combine sparse and dense features

The FeatureBuilder from `src/model.py` handles the full pipeline:
TF-IDF for text, label encoding for categoricals, and pass-through
for numerical features. It uses `scipy.sparse.hstack` to combine the
sparse TF-IDF matrix with the dense categorical and numerical arrays.

In [ ]:
X = fb.transform(df)
y = df['approved'].values.astype(int)

feature_names = fb.get_feature_names()
print(f"Combined feature matrix shape: {X.shape}")
print(f"Total feature names: {len(feature_names)}")
print(f"  TF-IDF features: {sum(1 for n in feature_names if n.startswith('tfidf__'))}")
print(f"  Categorical features: {sum(1 for n in feature_names if n in CATEGORICAL_COLS)}")
print(f"  Numerical features: {sum(1 for n in feature_names if n in NUMERICAL_COLS)}")

In [ ]:
# Check for NaN or infinite values in the combined matrix
n_nan = np.isnan(X).sum()
n_inf = np.isinf(X).sum()
print(f"NaN values in feature matrix: {n_nan}")
print(f"Inf values in feature matrix: {n_inf}")
print(f"Feature matrix dtype: {X.dtype}")

## 7. Feature value distributions

In [ ]:
# Distribution of the dense features (categoricals + numericals)
dense_feature_names = [n for n in feature_names if not n.startswith('tfidf__')]
dense_start = sum(1 for n in feature_names if n.startswith('tfidf__'))

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=dense_feature_names[:8]
)

for i, name in enumerate(dense_feature_names[:8]):
    r, c = divmod(i, 4)
    col_idx = dense_start + i
    fig.add_trace(
        go.Histogram(x=X[:, col_idx], nbinsx=30, name=name),
        row=r + 1, col=c + 1
    )

fig.update_layout(height=400, title_text='Dense feature distributions',
                  showlegend=False)
fig.show()

## 8. Approval rate by category and district

In [ ]:
# Top 10 categories by count with approval rate
cat_stats = df.groupby('category').agg(
    count=('approved', 'size'),
    approval_rate=('approved', 'mean')
).reset_index().sort_values('count', ascending=False).head(10)

fig = px.bar(
    cat_stats, x='category', y='count', color='approval_rate',
    color_continuous_scale='RdYlGn',
    title='Top 10 permit categories (color = approval rate)',
    labels={'category': 'Category', 'count': 'Number of permits'}
)
fig.update_layout(height=400)
fig.show()

## 9. Summary

The feature engineering pipeline produces a combined matrix with three
types of features:

| Type | Count | Method |
|---|---|---|
| TF-IDF text | 500 | Bigram TF-IDF on cleaned descriptions |
| Categorical | 5 | Label encoding of category, district, community, quadrant, permitted/discretionary |
| Temporal | 3 | Year, month, day-of-week from application date |

The sparse TF-IDF matrix and dense arrays are combined with `hstack`.
Class balance is acceptable for stratified splitting. The next notebook
trains and compares four classifiers on this feature matrix.